In [1]:

import os
from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd
from IPython.display import display
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import Binarizer, OneHotEncoder, PCA, SQLTransformer, VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# keep Spark on the project kernel's Python and local Java setup in VS Code.
JAVA_HOME = '/opt/homebrew/opt/openjdk@21/libexec/openjdk.jdk/Contents/Home'
os.environ['JAVA_HOME'] = JAVA_HOME
os.environ['PATH'] = f"{JAVA_HOME}/bin:{os.environ['PATH']}"
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'
os.environ['SPARK_LOCAL_HOSTNAME'] = 'localhost'
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# keep the pandas tables wide enough to read in the notebook.
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

# store the project paths, the watched stream folder, the stream checkpoint, and the shared seed in one place.
PROJECT_DIR = Path('/Users/alexdevoid/Documents/Stats/ST554-HW/FinalProject')

# start Spark session
spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('final_project_power_stream')
    .config('spark.ui.showConsoleProgress', 'false')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('ERROR')
spark.conf.set('spark.sql.shuffle.partitions', '8')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/27 23:47:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:

# read data from URL into pandas
power_pdf = pd.read_csv('https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv')

# convert into a cached Spark SQL DataFrame.
power_sdf = spark.createDataFrame(power_pdf).cache()
_ = power_sdf.count()

# summarize the row count, column count, and response column
data_summary = pd.DataFrame(
    {
        'rows': [power_pdf.shape[0]],
        'columns': [power_pdf.shape[1]],
        'response': ['Power_Zone_3'],
    }
)


# show summary
display(data_summary)
# how first rows of the Spark SQL DataFrame
display(power_pdf.head())

,rows,columns,response
0,47174,10,Power_Zone_3


,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0


In [3]:
dtype_summary = pd.DataFrame(power_sdf.dtypes, columns=['column', 'spark_type'])
dtype_summary

,column,spark_type
0,Temperature,double
1,Humidity,double
2,Wind_Speed,double
3,General_Diffuse_Flows,double
4,Diffuse_Flows,double
5,Power_Zone_1,double
6,Power_Zone_2,double
7,Power_Zone_3,double
8,Month,bigint
9,Hour,bigint


In [4]:
# cast Hour to double, shift Month to a zero-based index, and rename Power_Zone_3 to label
sql_transform = SQLTransformer(
    statement="""
    SELECT *,
           CAST(Hour AS DOUBLE) AS Hour_double,
           CAST(Month - 1 AS DOUBLE) AS Month_index,
           Power_Zone_3 AS label
    FROM __THIS__
    """
)

# binarize the cast Hour column using a 6.5 threshold
hour_binarizer = Binarizer(inputCol='Hour_double', outputCol='hour_binary', threshold=6.5)

# weather variables in one vector
weather_assembler = VectorAssembler(
    inputCols=['Temperature', 'Humidity', 'Wind_Speed', 'General_Diffuse_Flows', 'Diffuse_Flows'],
    outputCol='weather_features'
)

# reduce to two principal-component scores.
weather_pca = PCA(k=2, inputCol='weather_features', outputCol='weather_pcs')

# one-hot encode the zero-based Month index
month_encoder = OneHotEncoder(inputCols=['Month_index'], outputCols=['Month_vec'])

# combine
feature_assembler = VectorAssembler(
    inputCols=['weather_pcs', 'hour_binary', 'Power_Zone_1', 'Power_Zone_2', 'Month_vec'],
    outputCol='features'
)

# elastic-net linear regression model
elastic_net = LinearRegression(featuresCol='features', labelCol='label', predictionCol='prediction')

# wrap the transformations and model into a pipeline.
power_pipeline = Pipeline(
    stages=[
        sql_transform,
        hour_binarizer,
        weather_assembler,
        weather_pca,
        month_encoder,
        feature_assembler,
        elastic_net,
    ]
)

# build the grid
param_values = [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]
param_grid = (
    ParamGridBuilder()
    .addGrid(elastic_net.regParam, param_values)
    .addGrid(elastic_net.elasticNetParam, param_values)
    .build()
)

# use RMSE as the CV metric for the elastic-net fit.
rmse_evaluator = RegressionEvaluator(labelCol='label', predictionCol='prediction', metricName='rmse')


# 5-fold cross-validation over the pipeline.
RANDOM_STATE = 554
power_cv = CrossValidator(
    estimator=power_pipeline,
    estimatorParamMaps=param_grid,
    evaluator=rmse_evaluator,
    numFolds=5,
    parallelism=100,
    seed=RANDOM_STATE,
)

# fit the full cross-validated pipeline on the power data.
power_cv_model = power_cv.fit(power_sdf)

# find which tuning variable combination gave the smallest mean CV RMSE.
best_idx = int(np.argmin(power_cv_model.avgMetrics))
best_params = {param.name: value for param, value in param_grid[best_idx].items()}
cv_rmse = float(power_cv_model.avgMetrics[best_idx])

# score the data with the fitted model
training_predictions = power_cv_model.transform(power_sdf)
training_rmse = float(rmse_evaluator.evaluate(training_predictions))

# collect the tuning and error results into a table.
model_summary = pd.DataFrame(
    {
        'best_regParam': [float(best_params['regParam'])],
        'best_elasticNetParam': [float(best_params['elasticNetParam'])],
        'cv_rmse': [cv_rmse],
        'training_rmse': [training_rmse],
    }
)

display(model_summary)

,best_regParam,best_elasticNetParam,cv_rmse,training_rmse
0,0.05,0.5,2147.84652,2147.097346


In [5]:
# create residual column
residual_sdf = training_predictions.withColumn('residual', col('label') - col('prediction'))

# print a compact preview of the response, fitted value, and residual
residual_sdf.select('label', 'prediction', 'residual').show(20, truncate=False)

+-----------+------------------+------------------+
|label      |prediction        |residual          |
+-----------+------------------+------------------+
|20240.96386|20880.29948218109 |-639.33562218109  |
|20131.08434|18660.00907989481 |1471.07526010519  |
|19668.43373|18204.514233535734|1463.9194964642666|
|18899.27711|17590.430364613374|1308.8467453866251|
|18442.40964|16997.06083606251 |1445.3488039374934|
|18130.12048|16517.445990227792|1612.6744897722092|
|17945.06024|16093.011843798984|1852.0483962010148|
|17459.27711|15722.45502981135 |1736.8220801886491|
|17025.54217|15270.805408192911|1754.7367618070894|
|16794.21687|14938.101836262907|1856.1150337370927|
|16638.07229|14652.209161112158|1985.8631288878423|
|16395.18072|14414.725325693329|1980.4553943066712|
|16117.59036|14082.629442413661|2034.9609175863388|
|15822.6506 |13624.630222293086|2198.0203777069146|
|15672.28916|13450.1271513574  |2222.162008642601 |
|15597.10843|13302.087850397544|2295.0205796024566|
|15510.36145

In [ ]:
STREAM_INPUT_DIR = PROJECT_DIR / 'power_stream_input'

# reuse schema for the CSV stream
stream_schema = power_sdf.schema
raw_stream = spark.readStream.schema(stream_schema).option('header', True).csv(str(STREAM_INPUT_DIR))

# generate predictions with the fitted model
# add the residual column
stream_predictions = (
    power_cv_model.transform(raw_stream)
    .withColumn('residual', col('label') - col('prediction'))
    .select('label', 'prediction', 'residual')
)


# create a label column from the response
stream_labels = raw_stream.withColumnRenamed('Power_Zone_3', 'label').select('label')

# join stream transformations on the label column
joined_stream = stream_predictions.join(stream_labels, on='label')

# write the joined stream to the console and start query
stream_query = (
    joined_stream.writeStream
    .format('console')
    .outputMode('append')
    .option('truncate', False)
    .option('numRows', 20)
    .start()
)

print('stream active =', stream_query.isActive)



stream active = True


-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+------------------+-------------------+
|label      |prediction        |residual           |
+-----------+------------------+-------------------+
|9559.518072|10382.477799916855|-822.9597279168538 |
|10475.37994|13087.911190276165|-2612.5312502761644|
|10710.36145|12908.413741370416|-2198.0522913704153|
|22083.88664|21843.57441934123 |240.31222065877228 |
|9478.991597|7583.045314111058 |1895.9462828889418 |
|19246.26506|18145.754058903094|1100.5110010969074 |
|30178.30721|27789.81988631842 |2388.48732368158   |
|20200.72727|19522.331553349573|678.395716650426   |
|19727.55873|19816.028805387796|-88.47007538779508 |
|16768.40125|18284.506052909357|-1516.104802909358 |
|27145.84615|26312.148024953898|833.6981250461031  |
|10464.34574|7817.301040956118 |2647.0446990438822 |
|11824.2497 |10437.185288411249|1387.0644115887517 |
|18317.73279|18181.315739379217|136.41705062078108 |
|1